# Домашнее задание 3. Мониторинг ВМ и Docker-образ backend

Стенд развёрнут на Ubuntu Server 24.04 ARM64 в VMware Fusion. В первой части Node Exporter собирает метрики VM, Prometheus сохраняет их, а Grafana отображает собственный дашборд. Во второй части один backend упакован тремя способами: single-stage от root, multi-stage от root и multi-stage от непривилегированного пользователя.

## 1. Проверка окружения

In [1]:
%%bash
set -eu
hostnamectl | sed -n '1,8p'
docker --version
docker compose version
findmnt /mnt/hgfs

 Static hostname: devops
       Icon name: computer-vm
         Chassis: vm 🖴
      Machine ID: b

632bdc41b9e4187867dc9fd34cc6579
         Boot ID: 6efa68a797a74904bf787a86f8ff6d6d
  Virtualization:

 vmware
Operating System: Ubuntu 24.04.5 LTS
          Kernel: Linux 6.8.0-147-generic


Docker version 

29.1.3, build 29.1.3-0ubuntu3~24.04.2


Docker Compose version 2.40.3+ds1-0ubuntu1~24.04.1


TARGET    SOURCE      FSTYPE           OPTIONS
/mnt/hgfs vmhgfs-fuse fuse.vmhgfs-fuse rw,relatime,us

er_id=0,group_id=0,allow_other


## 2. Node Exporter → Prometheus → Grafana

Node Exporter работает в host network и читает файловые системы VM через `/host`. Prometheus обращается к нему через `host.docker.internal:9100`, а статическая метка `instance: vm1` используется переменной дашборда.

In [2]:
%%bash
set -eu
cd monitoring
echo '=== prometheus.yml ==='
cat prometheus/prometheus.yml
echo '=== compose services ==='
docker compose up -d
docker compose ps

=== prometheus.yml ===


global:
  scrape_interval: 15s        # как часто опрашивать цели
  evaluation

_interval: 15s    # как часто вычислять правила

scrape_configs:
  - job_nam

e: prometheus            # метрики самого Prometheus
    static_configs:
      - target

s: [localhost:9090]

  - job_name: node
    static_configs:
      - targets: [host.docker.internal:9

...

EATED         STATUS         PORTS
monitoring-grafana-1         grafana/grafana:12.1.0      "/run.sh

"                grafana         2 minutes ago   Up 2 minutes   127.0.0.1:3000->3000/tcp
monitoring-

node-exporter-1   prom/node-exporter:v1.9.1   "/bin/node_exporter …"   node-exporter   8 minutes a

go   Up 7 minutes   
monitoring-prometheus-1      prom/prometheus:v3.5.0      "/bin/prometheus --c

…"   prometheus      8 minutes ago   Up 7 minutes   127.0.0.1:9090->9090/tcp


In [3]:
%%bash
set -eu
cd monitoring
echo '=== promtool ==='
docker compose exec -T prometheus promtool check config /etc/prometheus/prometheus.yml </dev/null
echo '=== первые метрики node_* ==='
curl -s http://127.0.0.1:9100/metrics | grep -m 5 '^node_'
echo '=== активные цели Prometheus ==='
curl -s http://127.0.0.1:9090/api/v1/targets | python3 -c 'import json, sys; data=json.load(sys.stdin); print(*[{"job": t["labels"].get("job"), "instance": t["labels"].get("instance"), "health": t["health"]} for t in data["data"]["activeTargets"]], sep="\n")'

=== promtool ===


Checking /etc/prometheus/prometheus.yml


 SUCCESS: /etc/prometheus/prometheus.yml is valid prometheus config file syntax



=== первые метрики node_* ===


node_arp_entries{device="br-59a5a022b4fe"} 2
node_arp_entries{device="docker0"} 1
node_arp_entries{d

evice="enp2s0"} 3
node_boot_time_seconds 1.792857049e+09
node_context_switches_total 2.41242e+06


=== активные цели Prometheus ===


{'job': 'node', 'instance': 'vm1', 'health': 'up'}
{'job': 'prometheus', 'instance': 'localhost:9090

', 'health': 'up'}


### Grafana

Источник `Prometheus-vm1` подключён по внутреннему адресу Compose `http://prometheus:9090`. Динамическая переменная `instance` получает значения запросом `label_values(node_uname_info, instance)`.

Запросы четырёх панелей:

- CPU: `100 * (1 - avg by (instance) (rate(node_cpu_seconds_total{instance=~"$instance", mode="idle"}[$__rate_interval])))`
- RAM: `100 * (1 - node_memory_MemAvailable_bytes{instance=~"$instance"} / node_memory_MemTotal_bytes{instance=~"$instance"})`
- диск: `node_filesystem_avail_bytes{instance=~"$instance", fstype!~"tmpfs|overlay|squashfs"}`
- сеть: `rate(node_network_receive_bytes_total{instance=~"$instance", device!="lo"}[$__rate_interval])` и аналогичный запрос к `node_network_transmit_bytes_total`.

![Prometheus targets](screenshots/prometheus-targets.png)

![Grafana datasource](screenshots/grafana-datasource.png)

![Grafana dashboard](screenshots/grafana-dashboard.png)

In [4]:
%%bash
set -eu
echo '=== Save & test через API Grafana ==='
curl -s -u admin:admin http://127.0.0.1:3000/api/datasources/uid/prometheus-vm1/health
echo

=== Save & test через API Grafana ===


{"details":{"application":"Prometheus","features":{"rulerApiEnabled":false}},"message":"Successfully

 queried the Prometheus API.","status":"OK"}

In [ ]:
%%bash
set -eu
echo '=== Запросы панелей в dashboard.json ==='
grep -n '"expr"' monitoring/grafana/dashboard.json

=== Запросы панелей в dashboard.json ===
59:          "expr": "100 * (1 - avg by (instance) (rate(node_cpu_seconds_total{instance=~\"$instance\", mode=\"idle\"}[$__rate_interval])))",
103:          "expr": "100 * (1 - node_memory_MemAvailable_bytes{instance=~\"$instance\"} / node_memory_MemTotal_bytes{instance=~\"$instance\"})",
146:          "expr": "node_filesystem_avail_bytes{instance=~\"$instance\", fstype!~\"tmpfs|overlay|squashfs\"}",
189:          "expr": "rate(node_network_receive_bytes_total{instance=~\"$instance\", device!=\"lo\"}[$__rate_interval])",
196:          "expr": "rate(node_network_transmit_bytes_total{instance=~\"$instance\", device!=\"lo\"}[$__rate_interval])",


## 3. Три варианта Dockerfile

Во всех вариантах используется заданный базовый образ `python:3.11.6-slim-bookworm`. `Dockerfile.single` собирает и запускает приложение в одной стадии. Два остальных варианта собирают wheels в стадии `build`, а в финальный образ копируют только `dist`. В `Dockerfile.nonroot` пользователь `counter` создаётся через `addgroup` и `adduser` по примеру семинара.

In [5]:
%%bash
set -eu
cd counter-backend
for variant in single multistage nonroot; do
  echo "=== build ${variant} ==="
  docker build --file "Dockerfile.${variant}" --tag "counter-backend:${variant}" .
done

=== build single ===


DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            In

stall the buildx component to build images with BuildKit:
            https://docs.docker.com/go/bui

ldx/




Step 1/14 : FROM python:3.11.6-slim-bookworm


...

 ---> 6284e5d31b1c
Step 23/23 : ENTRYPOINT ["python", "-m", "counter_backend"]


 ---> Using cache


 ---> 1a98749dedc3


Successfully built 1a98749dedc3


Successfully tagged counter-backend:nonroot


### Проверка каждого образа

Для каждого варианта создаётся новый контейнер, поэтому первый GET обязан вернуть нулевой счётчик. После POST значение увеличивается. Команда `id` показывает пользователя главного процесса.

In [6]:
%%bash
set -eu
cd counter-backend
for variant in single multistage nonroot; do
  name="counter-${variant}"
  docker rm -f "${name}" >/dev/null 2>&1 || true
  echo "=== ${variant} ==="
  docker run --detach --name "${name}" --publish 127.0.0.1:8080:8080/tcp "counter-backend:${variant}"
  attempt=0
  until curl -fsS http://127.0.0.1:8080/api/counter >/dev/null 2>&1; do
    attempt=$((attempt + 1))
    test "${attempt}" -lt 30
    sleep 1
  done
  printf 'GET  '
  curl -sS -w ' status=%{http_code}\n' http://127.0.0.1:8080/api/counter
  printf 'POST '
  curl -sS -X POST -w ' status=%{http_code}\n' http://127.0.0.1:8080/api/counter
  printf 'GET  '
  curl -sS -w ' status=%{http_code}\n' http://127.0.0.1:8080/api/counter
  printf 'USER '
  docker exec "${name}" id
  docker rm --force "${name}" >/dev/null
done

=== single ===


50a0dfbb2b8573c38bb42f779393705aaa115f92ff5a8fb402f9798ec1e6a030


GET  

{"count": 0}

 status=200


...

 status=201
GET  

{"count": 1}

 status=200


USER 

uid=1001(counter) gid=1001(counter) groups=1001(counter),100(users)


## 4. Сравнение размеров

In [7]:
%%bash
docker image ls counter-backend --format 'table {{.Repository}}:{{.Tag}}\t{{.Size}}\t{{.ID}}'

REPOSITORY:TAG               SIZE      IMAGE ID
counter-backend:nonroot      261MB     1a98749dedc3


counter-backend:multistage   260MB     6d640643f6ff
counter-backend:single       391MB     30d8a8d29

506


## Итог

Node Exporter успешно опрашивается Prometheus как `vm1`, а Grafana показывает CPU, RAM, свободное место и сетевой трафик. Все три Docker-образа запускают backend и проходят функциональную проверку. Multi-stage варианты заметно меньше single-stage, а nonroot-вариант запускает процесс с UID 1001.